# AI Investment Committee Environment: minimal GRPO notebook

This notebook is the Round 2 training surface for the Portfolio Manager.

Recommended Colab account for this project: `anuagar@groww.in`

It does four things:
- installs the small TRL stack needed for a Colab smoke run
- checks heuristic and random baselines
- builds a LoRA-capable GRPO trainer against the committee environment
- saves a baseline report, raw training logs, and a reward-style curve

CPU runtimes are fine for the dry run and baseline checks. Actual GRPO training should be run on a GPU runtime.

Verified T4 smoke config: `Qwen/Qwen3-0.6B`, `use_lora=True`, `repeats_per_task=2`, `max_steps=4`.


In [ ]:
from pathlib import Path
import os
import sys

candidate_roots = [Path.cwd(), Path('/content/amc_allocator_env')]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'training').exists() and (candidate / 'tasks.py').exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError('Run this notebook from the amc_allocator_env repo root or clone the repo into /content/amc_allocator_env first.')

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('Repo root:', REPO_ROOT)

In [ ]:
!pip install -q trl datasets accelerate matplotlib peft jmespath "openenv-core[core]>=0.2.2" openai "git+https://github.com/huggingface/transformers.git@main"
# Optional acceleration layer if the runtime supports it:
# !pip install -q unsloth


In [ ]:
import json
from pathlib import Path
from pprint import pprint

import torch
from IPython.display import Image, Markdown, display

from training.committee_artifacts import (
    available_numeric_keys,
    export_baseline_report,
    export_judging_report,
    export_metric_series,
    export_onsite_demo_summary,
    export_training_log_history,
    plot_metric_series,
)
from training.committee_eval import evaluate_policy, summarize_rows
from training.committee_grpo_train import build_trainer, print_baseline_summary

GPU_AVAILABLE = torch.cuda.is_available()
print('CUDA available:', GPU_AVAILABLE)
if not GPU_AVAILABLE:
    print('CPU runtime detected. Use this notebook for dry-run and baseline validation only.')


def render_summary_table(summary_rows):
    columns = [
        ('Policy', 'policy'),
        ('Score', 'score'),
        ('Return', 'total_return'),
        ('Drawdown', 'max_drawdown'),
        ('Compliance', 'compliance_score'),
        ('Info Usage', 'information_usage'),
        ('Risk Response', 'risk_response'),
    ]
    header = '| ' + ' | '.join(label for label, _ in columns) + ' |'
    divider = '| ' + ' | '.join(['---'] * len(columns)) + ' |'
    lines = [header, divider]
    for row in summary_rows:
        cells = []
        for _, key in columns:
            value = row[key]
            if isinstance(value, float):
                cells.append(f'{value:.4f}')
            else:
                cells.append(str(value))
        lines.append('| ' + ' | '.join(cells) + ' |')
    display(Markdown('\n'.join(lines)))


def render_markdown_file(path, heading=None):
    if heading:
        display(Markdown(heading))
    display(Markdown(Path(path).read_text()))


In [ ]:
print_baseline_summary()

EVAL_SEEDS = (7, 11, 13)
heuristic_rows = evaluate_policy('heuristic', seeds=EVAL_SEEDS)
random_rows = evaluate_policy('random', seeds=EVAL_SEEDS)

heuristic_summary = summarize_rows(heuristic_rows)
random_summary = summarize_rows(random_rows)

print('\nHeuristic overall:')
pprint(heuristic_summary)
print('\nRandom overall:')
pprint(random_summary)

display(Markdown('## Baseline snapshot'))
render_summary_table([heuristic_summary, random_summary])
display(
    Markdown(
        f"**Score delta vs random:** {heuristic_summary['score'] - random_summary['score']:.4f}  \\n"
        f"**Return delta vs random:** {heuristic_summary['total_return'] - random_summary['total_return']:.4f}"
    )
)


In [ ]:
MODEL_NAME = 'Qwen/Qwen3-0.6B'
COLAB_ACCOUNT_EMAIL = 'anuagar@groww.in'
OUTPUT_DIR = 'outputs/committee-grpo-onsite'
USE_LORA = True
LORA_TARGET_MODULES = ['q_proj', 'v_proj']
REPEATS_PER_TASK = 2 if GPU_AVAILABLE else 1
MAX_STEPS = 4 if GPU_AVAILABLE else 1

trainer = build_trainer(
    model_name=MODEL_NAME,
    output_dir=OUTPUT_DIR,
    repeats_per_task=REPEATS_PER_TASK,
    max_steps=MAX_STEPS,
    use_lora=USE_LORA,
    lora_target_modules=LORA_TARGET_MODULES,
)
print('Trainer ready:', type(trainer).__name__)
print('Dataset rows:', len(trainer.train_dataset))
if not GPU_AVAILABLE:
    print('Stop here on CPU. Switch Colab to a GPU runtime before running the training cell.')


In [ ]:
if not GPU_AVAILABLE:
    raise RuntimeError('GPU runtime required for actual GRPO training. Switch Runtime > Change runtime type and rerun from Cell 5.')

train_output = trainer.train()
trainer.save_model(OUTPUT_DIR)

baseline_report = export_baseline_report(OUTPUT_DIR)
baseline_payload = json.loads(Path(baseline_report).read_text())
training_log = export_training_log_history(OUTPUT_DIR, trainer.state.log_history)
judging_report = export_judging_report(
    OUTPUT_DIR,
    model_name=MODEL_NAME,
    colab_account_email=COLAB_ACCOUNT_EMAIL,
    baseline_payload=baseline_payload,
    log_history=trainer.state.log_history,
    notes='Minimal GRPO smoke run for Round 2 committee environment with LoRA adapters.',
)
demo_summary = export_onsite_demo_summary(
    OUTPUT_DIR,
    model_name=MODEL_NAME,
    colab_account_email=COLAB_ACCOUNT_EMAIL,
    baseline_payload=baseline_payload,
    log_history=trainer.state.log_history,
    notes='Use this file as the judge-facing one-pager during the onsite demo.',
)

print(train_output)
print('Baseline report:', baseline_report)
print('Training log:', training_log)
print('Judging report JSON:', judging_report[0])
print('Judging report Markdown:', judging_report[1])
print('Onsite demo summary:', demo_summary)
print('Numeric log keys:', available_numeric_keys(trainer.state.log_history))



In [ ]:
series_path = export_metric_series(OUTPUT_DIR, trainer.state.log_history)
plot_path = plot_metric_series(trainer.state.log_history, OUTPUT_DIR)

print('Series JSON:', series_path)
print('Curve PNG:', plot_path)
display(Image(filename=str(plot_path)))

display(Markdown('## Inline artifact preview'))
render_markdown_file(judging_report[1], heading='### Judging report')
render_markdown_file(demo_summary, heading='### Onsite demo summary')



## Submission-ready evidence

This notebook is the reproducible training run for the Portfolio Manager in the AI Investment Committee environment.

After the final cells run, keep these four artifacts from `outputs/committee-grpo-onsite/`:
- `baseline_report.json`: random versus heuristic PM scores across all committee tasks.
- `training_log_history.json`: raw TRL/GRPO trainer log history from the run.
- `judging_report.md`: compact summary of model, baseline deltas, and reward progress.
- `reward_curve.png`: the visual proof that the smoke run produced measurable reward signal.

The claim to make from this notebook is intentionally narrow: the environment is trainable and the verifier rewards conflict-aware behavior. It is not claiming full convergence or a production trading agent.

For the final submission story, connect this notebook to the README figures: conflict snapshot -> baseline comparison -> GRPO reward curve -> anti-hack probe table.